# Benchmark Scenario 004 — CSQ trên 4 Dataset\n
\n
**KB4**: So sánh framework CSQ với backbone ViT-B_32 / bit=32 / epoch=150 trên 4 dataset: cifar10, coco, nuswide_21, imagenet.\n
\n
## Workflow\n
1. Chạy **Cell 1–4** (mount Drive → clone repo → cài deps → symlink) mỗi khi mở Colab runtime mới.\n
2. Chạy **Cell 5** (config) — đổi `EPOCH=10, TEST_MAP=5` cho smoke test local, `EPOCH=150, TEST_MAP=30` cho Colab.\n
3. Mở **4 runtime Colab song song**, mỗi runtime chạy 1 trong Cell 6–9 để rút ngắn wall-clock.\n
\n
Kết quả ghi vào `Checkpoints_Results/CSQ-ViT-B_32-{dataset}/` (persist qua Drive symlink).

In [ ]:
# Cell 1 — Mount Google Drive\n
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
# Cell 2 — Clone repo và chuyển vào thư mục dự án\n
!git clone https://github.com/hatuan314/VisionTransformerHashing.git
%cd VisionTransformerHashing

In [ ]:
# Cell 3 — Cài dependency thiếu trên Colab\n
!pip install ml_collections

In [ ]:
# Cell 4 — Tạo symlink tới Drive (pretrainedVIT, dataset, Checkpoints_Results)\n
import os

DRIVE_BASE = '/content/gdrive/MyDrive/master_is/semester_3/IR/VTS-LAB'

for link in ['pretrainedVIT', 'dataset', 'Checkpoints_Results']:
    if os.path.islink(link):
        os.remove(link)

os.symlink(f'{DRIVE_BASE}/pretrainedVIT', 'pretrainedVIT')
os.symlink(f'{DRIVE_BASE}/dataset', 'dataset')

os.makedirs(f'{DRIVE_BASE}/Checkpoints_Results', exist_ok=True)
os.symlink(f'{DRIVE_BASE}/Checkpoints_Results', 'Checkpoints_Results')

!ls pretrainedVIT/
!ls dataset/
!ls Checkpoints_Results/

In [ ]:
# Cell 4b — Download COCO 2014 vào Colab local (bỏ qua Drive, tránh tốn quota)
# Chạy cell này TRƯỚC Cell 7. Mỗi lần runtime mới phải chạy lại (~30-40 phút).
import os, subprocess

REPO = '/content/VisionTransformerHashing'
COCO_DIR = f'{REPO}/dataset/COCO'

# Nếu symlink Cell 4 đang trỏ Drive → tháo ra, tạo thư mục local thay thế
dataset_link = f'{REPO}/dataset'
if os.path.islink(dataset_link):
    drive_dataset = os.readlink(dataset_link)
    os.remove(dataset_link)
    os.makedirs(dataset_link, exist_ok=True)
    # Giữ lại cifar10/imagenet/nuswide nếu đã có trên Drive
    for sub in os.listdir(drive_dataset):
        if sub != 'COCO':
            os.symlink(f'{drive_dataset}/{sub}', f'{dataset_link}/{sub}')

os.makedirs(f'{COCO_DIR}/train2014', exist_ok=True)
os.makedirs(f'{COCO_DIR}/val2014',   exist_ok=True)

URLS = [
    ('http://images.cocodataset.org/zips/train2014.zip', COCO_DIR),
    ('http://images.cocodataset.org/zips/val2014.zip',   COCO_DIR),
]
for url, dest in URLS:
    fname = url.split('/')[-1]
    # Kiểm tra xem đã có ảnh chưa (không cần download lại nếu đã giải nén)
    folder = fname.replace('.zip', '')
    imgs = os.listdir(f'{dest}/{folder}')
    if len(imgs) > 100:
        print(f'{folder}: {len(imgs)} ảnh — bỏ qua download.')
        continue
    print(f'Downloading {fname} (~', end='', flush=True)
    print('13GB' if 'train' in fname else '6GB', end=')... ', flush=True)
    subprocess.run(['wget', '-q', '--show-progress', '-P', dest, url], check=True)
    print(f'Giải nén {fname}...')
    subprocess.run(['unzip', '-q', f'{dest}/{fname}', '-d', dest], check=True)
    os.remove(f'{dest}/{fname}')
    print(f'{fname} xong.')

print('COCO sẵn sàng:', os.listdir(COCO_DIR))


In [ ]:
# Cell 5 — Config\n
# smoke local: đổi EPOCH=10, TEST_MAP=5\n
EPOCH = 150
TEST_MAP = 30

In [ ]:
# Cell 6 — Train CSQ trên cifar10 (chạy trên 1 Colab runtime riêng)
!python CSQ.py --dataset cifar10 --bit 32 --epoch {EPOCH} --test_map {TEST_MAP} --backbone ViT-B_32 --save_path Checkpoints_Results/CSQ-ViT-B_32-cifar10

In [ ]:
# Cell 7 — Train CSQ trên coco (chạy trên 1 Colab runtime riêng)
!python CSQ.py --dataset coco --bit 32 --epoch {EPOCH} --test_map {TEST_MAP} --backbone ViT-B_32 --save_path Checkpoints_Results/CSQ-ViT-B_32-coco

In [ ]:
# Cell 8 — Train CSQ trên nuswide_21 (chạy trên 1 Colab runtime riêng)
!python CSQ.py --dataset nuswide_21 --bit 32 --epoch {EPOCH} --test_map {TEST_MAP} --backbone ViT-B_32 --save_path Checkpoints_Results/CSQ-ViT-B_32-nuswide_21

In [ ]:
# Cell 9 — Train CSQ trên imagenet (chạy trên 1 Colab runtime riêng)\n
!python CSQ.py --dataset imagenet --bit 32 --epoch {EPOCH} --test_map {TEST_MAP} --backbone ViT-B_32 --save_path Checkpoints_Results/CSQ-ViT-B_32-imagenet